# RBPM — complete research run with live progress and ETA
Fresh Colab CPU runtime; select Runtime → Run all. A preflight and independent CPU benchmark precede the simulations. The overall bar counts computational tasks; stage bars report within-stage elapsed time and ETA. The overall compute ETA is adaptive and excludes plot/export overhead. The final cell downloads results.zip.


In [1]:
# Install required libraries.
%pip -q install "numba>=0.60" "arviz>=0.20" "matplotlib>=3.9" "tqdm>=4.66"


In [2]:
MODE = "research" #@param ["quick", "research"]
SEED_RUN = 20260816


In [3]:
"""Reproducible RBPM experiments. Python 3.10+, CPU; no external data downloads."""
import argparse, json, math, shutil, time, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import gammaln, expit, logit
from scipy.optimize import minimize
from scipy.stats import t, norm
from sklearn.datasets import load_wine
from sklearn.metrics import roc_auc_score
from numba import njit
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import arviz as az
from tqdm.auto import tqdm

AL, AU = .55, 1.5
NAMES = ['mu','theta','epsilon','alpha']
METHODS = ['SV','CPM','GBPM','MAD','IQR','Student-t','RBPM-MAP']
COLORS = ['#666666','#0072B2','#56B4E9','#E69F00','#CC79A7','#009E73','#D55E00']
SEED=20260816
@njit
def logistic(x):
    if x>=0: return 1/(1+math.exp(-x))
    z=math.exp(x); return z/(1+z)
@njit
def constants(a):
    lb=a*(math.lgamma(3/(2*a))-math.lgamma(1/(2*a)))
    return lb, math.log(a)+lb/(2*a)-math.lgamma(1/(2*a))
@njit
def logadd(x,y):
    m=max(x,y)
    return m+math.log1p(math.exp(-abs(x-y)))
@njit
def target(q,x,lg,loc,refvar,jac,continuous):
    mu,lv,le,h=q
    if abs(lv)>45 or abs(le)>35 or abs(mu)>1e5: return -np.inf
    e=logistic(le)
    if continuous:
        if abs(h)>35: return -np.inf
        u=logistic(h); a=AL+(AU-AL)*u
    else:
        a=h
        if a<AL or a>AU: return -np.inf
    lb,lc=constants(a); b0=4*refvar**a
    lp=-.5*((mu-loc)/5)**2 + 5*math.log(b0)-6*lv-b0*math.exp(-lv) + .5*math.log(e)+7*math.log1p(-e)
    if jac: lp+=lv+math.log(e)+math.log1p(-e)+math.log(u)+math.log1p(-u)
    clean0=math.log1p(-e)+lc-lv/(2*a); cont0=math.log(e)
    scale=math.exp(lb-lv)
    for i in range(x.size):
        f=clean0-scale*abs(x[i]-mu)**(2*a)
        lp+=logadd(f,cont0+lg[i])
    return lp
@njit
def chains(x,lg,loc,refvar,start,warm,draws,seed):
    np.random.seed(seed)
    out=np.empty((4,draws,4)); acceptance=np.zeros((4,4))
    for c in range(4):
        q=start.copy(); q[0]+=np.random.normal()*math.sqrt(refvar)*.4
        q[1]+=np.random.normal()*.6; q[2]+=np.random.normal()*.8; q[3]+=np.random.normal()*.8
        cur=target(q,x,lg,loc,refvar,True,True)
        step=np.array([.18*math.sqrt(refvar),.28,.6,.8]); window=np.zeros(4)
        for k in range(warm+draws):
            for j in range(4):
                old=q[j]; q[j]=old+step[j]*np.random.normal()
                cand=target(q,x,lg,loc,refvar,True,True)
                if math.log(np.random.random())<cand-cur:
                    cur=cand
                    if k<warm: window[j]+=1
                    else: acceptance[c,j]+=1
                else: q[j]=old
            if k<warm and (k+1)%50==0:
                rate=0.7/math.sqrt((k+1)/50)
                for j in range(4):
                    step[j]=min(5.,max(.001,step[j]*math.exp(rate*(window[j]/50-.44))))
                    window[j]=0
            if k>=warm:
                a=AL+(AU-AL)*logistic(q[3]); out[c,k-warm,0]=q[0];out[c,k-warm,1]=math.exp(q[1]/a);out[c,k-warm,2]=logistic(q[2]);out[c,k-warm,3]=a
    return out,acceptance/draws

def g_log(x,loc,scale): return t.logpdf(x,3,loc=loc,scale=scale)
def responsibilities(x,draw,g_loc,g_scale):
    mu,theta,e,a=np.asarray(draw).T
    mu,theta,e,a=[v[...,None] for v in (mu,theta,e,a)]
    lb=a*(gammaln(3/(2*a))-gammaln(1/(2*a))); lc=np.log(a)+lb/(2*a)-gammaln(1/(2*a))
    lf=lc-.5*np.log(theta)-np.exp(lb)*(np.abs(x-mu)/np.sqrt(theta))**(2*a)
    return expit(np.log(e)-np.log1p(-e)+g_log(x,g_loc,g_scale)-lf)
def finite_objective(q,x,lg,loc,refvar):
    value = -target(q,x,lg,loc,refvar,False,False)
    return float(value) if np.isfinite(value) else 1e100

def mapfit(x,loc=0,scale=5,refvar=1):
    lg=g_log(x,loc,scale); med=np.median(x); mad=max(.05,1.4826*np.median(abs(x-med)))
    starts=[np.array([med,np.log(mad**2),logit(.12),1.]),np.array([np.mean(x),np.log(max(np.var(x),.01)),logit(.05),.7])]
    bounds=[(-1e5,1e5),(-20,20),(-16,10),(AL,AU)]
    opts=[minimize(lambda q:finite_objective(q,x,lg,loc,refvar),v,method='L-BFGS-B',bounds=bounds,options={'maxiter':150,'ftol':1e-10}) for v in starts]
    valid=[o for o in opts if np.isfinite(o.fun) and o.fun<1e99]
    if not valid: raise RuntimeError('MAP optimization failed for all starts')
    best=min(valid,key=lambda o:o.fun); m,lv,le,a=best.x
    return np.array([m,np.exp(lv/a),expit(le),a]),bool(best.success)
def baseline(x,refvar=1):
    a=.75; n=len(x); y=np.abs(x-x.mean())**(2*a); m0=2**a*math.exp(gammaln(a+.5)-gammaln(.5)); an=5+n/(2*a); bn=4*refvar**a+y.sum()/(2*a*m0)
    mad=(np.median(abs(x-np.median(x)))/norm.ppf(.75))**2; iqr=((np.quantile(x,.75)-np.quantile(x,.25))/(2*norm.ppf(.75)))**2
    opt=minimize(lambda q:-np.sum(t.logpdf(x,3,loc=q[0],scale=np.exp(q[1]))),[np.median(x),np.log(max(np.std(x),.02))],method='L-BFGS-B',bounds=[(None,None),(-12,12)])
    return [np.var(x,ddof=1),(y.mean()/m0)**(1/a),math.exp(np.log(bn)/a+gammaln(an-1/a)-gammaln(an)),mad,iqr,3*np.exp(2*opt.x[1])]
def sample(rng,n,a,e,kind):
    lb,_=constants(a); x=rng.choice([-1.,1.],n)*(rng.gamma(1/(2*a),size=n)/np.exp(lb))**(1/(2*a)); z=rng.random(n)<e
    if kind=='matched': g=5*rng.standard_t(3,n)
    elif kind=='shifted-t': g=6+np.sqrt(4/3)*rng.standard_t(3,n)
    elif kind=='shifted-normal': g=rng.normal(6,2,n)
    else: g=rng.normal(size=n)
    x[z]=g[z]; return x,z.astype(int)
def metrics(z,p): return (roc_auc_score(z,p) if np.unique(z).size==2 else np.nan, np.mean((z-p)**2))
def diagnostic_scalar(value, name):
    # ArviZ may return a scalar, a 0-d array, or a singleton array/DataArray.
    arr = np.asarray(value, dtype=float)
    if arr.size != 1:
        raise ValueError(f"{name}: expected one diagnostic for one parameter, got shape {arr.shape}")
    return arr.item()

def posterior(x,loc,scale,refvar,warm,draws,seed,fit=None):
    if fit is None: fit,_=mapfit(x,loc,scale,refvar)
    m,th,e,a=fit; q=np.array([m,a*np.log(th),logit(e),logit(np.clip((a-AL)/(AU-AL),.03,.97))])
    ar,acc=chains(x,g_log(x,loc,scale),loc,refvar,q,warm,draws,int(seed)); flat=ar.reshape(-1,4)
    p=np.zeros(len(x))
    for block in np.array_split(flat,max(1,len(flat)//500)): p+=responsibilities(x,block,loc,scale).sum(axis=0)
    p/=len(flat)
    rows=[]
    for j,name in enumerate(NAMES):
        v=ar[:,:,j]; rh=diagnostic_scalar(az.rhat(v,method='rank'),'rank Rhat'); eb=diagnostic_scalar(az.ess(v,method='bulk'),'bulk ESS'); et=diagnostic_scalar(az.ess(v,method='tail',prob=.05),'tail ESS'); mc=diagnostic_scalar(az.mcse(v,method='mean'),'mean MCSE')
        rows.append(dict(parameter=name,mean=v.mean(),median=np.median(v),lo=np.quantile(v,.025),hi=np.quantile(v,.975),rhat=rh,ess_bulk=eb,ess_tail=et,mcse=mc,acceptance=acc[:,j].mean(),diagnostic_pass=bool(np.isfinite(rh) and rh<=1.01 and min(eb,et)>=400)))
    return ar,p,pd.DataFrame(rows)

def risk_summary(df,keys):
    d=df.copy();d['error']=d.estimate-d.truth;d['sq']=d.error**2;d['abs']=abs(d.error)
    g=d.groupby(keys,sort=False,dropna=False); out=g.agg(reps=('error','size'),bias=('error','mean'),mse=('sq','mean'),mae=('abs','mean'),median_abs_error=('abs','median'),bias_mcse=('error','sem'),mse_mcse=('sq','sem')).reset_index();out['rmse']=np.sqrt(out.mse);out['rmse_mcse']=out.mse_mcse/(2*out.rmse)
    if 'kind' in out:
        heavy=out.kind.isin(['matched','shifted-t']) & (out.epsilon>0)
        out.loc[heavy,['rmse_mcse','mse_mcse']]=np.nan
    return out

def setup_plot():
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.18,'figure.dpi':130,'savefig.dpi':250,'pdf.fonttype':42,'legend.frameon':False})
def save(fig,out,name):
    fig.tight_layout(rect=(0,.13,1,1) if name in ('risk','median_error') else None);fig.savefig(out/'figures'/f'{name}.pdf',bbox_inches='tight');fig.savefig(out/'figures'/f'{name}.png',bbox_inches='tight');plt.close(fig)
def tab(df,out,name):
    df.to_csv(out/'tables'/f'{name}.csv',index=False)
def tex_table(df,cols,headers=None):
    v=df[cols].copy();v.columns=headers or cols
    return v.to_latex(index=False,float_format=lambda x:f'{x:.3f}',escape=True,na_rep='--')
def figtex(name,caption): return '\\begin{figure}[htbp]\n\\centering\n\\includegraphics[width=.92\\linewidth]{results/figures/'+name+'.pdf}\n\\caption{'+caption+'}\\label{fig:new-'+name+'}\n\\end{figure}\n'
def tabletex(df,cols,caption,headers=None): return '\\begin{table}[htbp]\n\\centering\\small\n\\caption{'+caption+'}\n'+tex_table(df,cols,headers)+'\\end{table}\n'


def format_duration(seconds):
    if not np.isfinite(seconds): return 'estimating...'
    seconds=max(0,int(round(seconds)))
    h,r=divmod(seconds,3600); m,sec=divmod(r,60)
    return f'{h:d}h {m:02d}m {sec:02d}s' if h else f'{m:d}m {sec:02d}s'

class RunProgress:
    """Stage-level tqdm plus overall ETA based on measured task-type costs."""
    def __init__(self, specs, rates):
        self.specs=specs; self.rates=rates.copy(); self.remaining={k:0 for k in rates}
        for _,n,k in specs: self.remaining[k]+=n
        self.total=sum(n for _,n,_ in specs); self.count=0; self.start=time.perf_counter()
        self.bar=tqdm(total=self.total,desc='Overall computational tasks',unit='fit/rep',
                      dynamic_ncols=True,bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}{postfix}]')
        self.stage_bar=None; self.key=None; self.last=None; self.last_render=0
        self.refresh(force=True)
    def refresh(self,force=False):
        now=time.perf_counter()
        if force or now-self.last_render>=0.8:
            eta=sum(self.remaining[k]*self.rates[k] for k in self.rates)
            self.bar.set_postfix_str('compute ETA '+format_duration(eta),refresh=True)
            self.last_render=now
    def stage(self,i):
        if self.stage_bar is not None: self.stage_bar.close()
        label,n,self.key=self.specs[i]
        self.stage_bar=tqdm(total=n,desc=label,unit='rep',dynamic_ncols=True,leave=True)
        self.last=time.perf_counter(); self.refresh(force=True)
    def tick(self):
        now=time.perf_counter(); dt=now-self.last; self.last=now
        self.rates[self.key]=.95*self.rates[self.key]+.05*dt
        self.remaining[self.key]-=1; self.count+=1
        self.stage_bar.update(1); self.bar.update(1); self.refresh(force=self.stage_bar.n==self.stage_bar.total)
    def skip_stage(self):
        n=self.stage_bar.total-self.stage_bar.n
        self.remaining[self.key]-=n; self.count+=n
        self.stage_bar.update(n); self.bar.update(n); self.refresh(force=True)
    def detail(self, text):
        self.stage_bar.set_postfix_str(text,refresh=True)
        self.last=time.perf_counter()
    def finish(self):
        if self.stage_bar is not None: self.stage_bar.close()
        self.refresh(force=True); self.bar.close()
        print('All computational tasks complete. Finalizing figures, tables and ZIP.',flush=True)

def benchmark_progress(warm,draws,seed):
    # Independent random stream: benchmark does not alter any scientific result.
    brng=np.random.default_rng(seed+91731); costs={'risk':[],'posterior':[],'wine':[]}
    print('Benchmarking CPU speed for an initial runtime estimate...',flush=True)
    x0,z0=sample(brng,80,1.,.1,'matched'); posterior(x0,0,5,1,200,250,int(brng.integers(1,2**31-1)))
    for n in (100,300):
        x,z=sample(brng,n,1.,.1,'matched')
        tic=time.perf_counter(); fit,_=mapfit(x); baseline(x); metrics(z,responsibilities(x,fit,0,5)); costs['risk'].append(time.perf_counter()-tic)
        tic=time.perf_counter(); posterior(x,0,5,1,warm,draws,int(brng.integers(1,2**31-1))); costs['posterior'].append(time.perf_counter()-tic)
    w=load_wine(); v=w.data[:,w.feature_names.index('flavanoids')]
    for cls in (1,2,2):
        x=np.r_[brng.choice(v[w.target==0],50),brng.choice(v[w.target==cls],8)]
        tic=time.perf_counter(); fit,_=mapfit(x,2.5,2,.25); baseline(x,.25); responsibilities(x,fit,2.5,2); costs['wine'].append(time.perf_counter()-tic)
    rates={k:max(.001,float(np.median(v))) for k,v in costs.items()}
    print('Measured seconds per task (initial): '+', '.join(f'{k}={v:.3f}' for k,v in rates.items()),flush=True)
    return rates

def run(mode,out,seed,resume_risk=False):
    start=time.time(); out=Path(out); (out/'figures').mkdir(parents=True,exist_ok=True);(out/'tables').mkdir(exist_ok=True);(out/'raw').mkdir(exist_ok=True)
    cfg={'quick':(4,2,10,200,250),'research':(200,60,300,1000,2000)}[mode]; nr,ncov,nboot,warm,draws=cfg
    rng=np.random.default_rng(seed); nextseed=lambda:int(rng.integers(1,2**31-1)); setup_plot()
    # Validate density normalization and variance numerically.
    from scipy.integrate import quad
    checks=[]
    for a in [.55,.7,1.,1.3,1.5]:
        lb,lc=constants(a); f=lambda x:np.exp(lc-np.exp(lb)*abs(x)**(2*a)); mass=2*quad(f,0,np.inf)[0];var=2*quad(lambda x:x*x*f(x),0,np.inf)[0]
        assert abs(mass-1)<1e-7 and abs(var-1)<1e-7;checks.append(dict(alpha=a,mass=mass,variance=var))
    tab(pd.DataFrame(checks),out,'density_checks')
    assert abs(constants(1.)[0]-np.log(.5))<1e-12
    probe=np.array([0.,1.,.1,1.]); xx=np.array([0.,2.,20.]); expected=.1*t.pdf(xx,3,scale=5)/(.9*norm.pdf(xx)+.1*t.pdf(xx,3,scale=5));assert np.allclose(responsibilities(xx,probe,0,5),expected)
    q=np.array([0.,0.,logit(.1),logit((1.-AL)/(AU-AL))]); direct=np.array([0.,0.,logit(.1),1.]); jac_expected=np.log(.1*.9)+np.log((1.-AL)/(AU-AL)*(AU-1.)/(AU-AL));assert np.allclose(target(q,xx,g_log(xx,0,5),0,1,True,True)-target(direct,xx,g_log(xx,0,5),0,1,False,False),jac_expected)
    scenarios=[('matched',a,n,e) for a in [.7,1.,1.3] for n in [100,300] for e in [0.,.1,.2]]+[(k,1.,n,e) for k in ['shifted-t','shifted-normal','overlap'] for n in [100,300] for e in [.1,.2]]
    specs=[('1/6 Risk simulations',len(scenarios)*nr,'risk'),('2/6 Posterior coverage',5*ncov,'posterior'),('3/6 Extreme-observation analysis',7,'posterior'),('4/6 Wine primary posterior',1,'posterior'),('5/6 Wine sensitivity',9,'posterior'),('6/6 Wine resampling',2*nboot,'wine')]
    rates=benchmark_progress(warm,draws,seed)
    print('Initial estimated computational runtime: '+format_duration(sum(n*rates[k] for _,n,k in specs))+' (excludes plotting, setup and ZIP; adapts during execution)',flush=True)
    progress=RunProgress(specs,rates); progress.stage(0)
    risk_path=out/'raw/risk.csv'; alloc_path=out/'raw/allocations.csv'
    risk_reused=False
    if resume_risk and risk_path.exists() and alloc_path.exists():
        risk=pd.read_csv(risk_path); alloc=pd.read_csv(alloc_path)
        keys=['kind','alpha','n','epsilon','rep']
        expected={(k,a,n,e,r) for k,a,n,e in scenarios for r in range(nr)}
        actual=set(alloc[keys].itertuples(index=False,name=None))
        actual_risk=set(risk[keys+['method']].itertuples(index=False,name=None))
        expected_risk={(*key,m) for key in expected for m in METHODS}
        if actual!=expected or len(alloc)!=len(expected) or actual_risk!=expected_risk or len(risk)!=len(expected_risk):
            raise ValueError('Saved risk files do not match this mode/scenario design. Preserve them separately and run with resume_risk=False.')
        if not np.isfinite(risk[['estimate','truth']].to_numpy()).all():
            raise ValueError('Non-finite saved risk estimates; inspect before reuse.')
        meta_path=out/'raw/risk_checkpoint.json'
        if meta_path.exists():
            meta=json.loads(meta_path.read_text())
            if meta['seed']!=seed or meta['mode']!=mode: raise ValueError('Checkpoint seed/mode mismatch.')
            rng.bit_generator.state=meta['rng_state']
        else:
            # Legacy notebook: risk stage used random draws only inside sample().
            # Caller must confirm identical seed and mode to resume legacy CSVs.
            for kind,a,n,e in scenarios:
                for rep in range(nr): sample(rng,n,a,e,kind)
        risk_reused=True
        print(f'Resumed {len(alloc)} saved risk replications; no risk optimization repeated.',flush=True)
        progress.skip_stage()
    else:
        risk=[]; alloc=[]
        for kind,a,n,e in scenarios:
            progress.detail(f'{kind}, a={a:g}, n={n}, eps={e:g}')
            for rep in range(nr):
                x,z=sample(rng,n,a,e,kind); fit,ok=mapfit(x); p=responsibilities(x,fit,0,5); auc,brier=metrics(z,p)
                common=dict(kind=kind,alpha=a,n=n,epsilon=e,rep=rep)
                for name,val in zip(METHODS,baseline(x)+[fit[1]]):risk.append(dict(**common,method=name,estimate=val,truth=1.))
                alloc.append(dict(**common,alpha_hat=fit[3],epsilon_hat=fit[2],auc=auc,brier=brier,optimizer_success=ok))
                progress.tick()
        risk=pd.DataFrame(risk);alloc=pd.DataFrame(alloc);risk.to_csv(out/'raw/risk.csv',index=False);alloc.to_csv(out/'raw/allocations.csv',index=False)
    (out/'raw/risk_checkpoint.json').write_text(json.dumps(dict(seed=seed,mode=mode,rng_state=rng.bit_generator.state)))
    rs=risk_summary(risk,['kind','alpha','n','epsilon','method']);tab(rs,out,'risk_summary'); ag=alloc.groupby(['kind','alpha','n','epsilon']).agg(auc=('auc','mean'),brier=('brier','mean'),alpha_hat=('alpha_hat','mean'),epsilon_hat=('epsilon_hat','mean'),optimizer_success=('optimizer_success','mean')).reset_index();tab(ag,out,'allocation_summary')
    fig,axes=plt.subplots(2,3,figsize=(12,7),sharex=True)
    for ax,(kind,a) in zip(axes.flat,[('matched',.7),('matched',1.),('matched',1.3),('shifted-t',1.),('shifted-normal',1.),('overlap',1.)]):
        sub=rs[(rs.kind==kind)&(rs.alpha==a)&(rs.n==300)]
        for m,c in zip(METHODS,COLORS):
            d=sub[sub.method==m];ax.errorbar(d.epsilon,d.rmse,yerr=d.rmse_mcse.fillna(0),marker='o',ms=3,color=c,label=m,lw=1.4)
        ax.set_title(f'{kind}, alpha={a:g}');ax.set_xlabel('Contamination probability');ax.set_yscale('symlog',linthresh=.2);ax.set_ylabel('RMSE (variance units)')
    handles,labels=axes[0,0].get_legend_handles_labels();fig.legend(handles,labels,loc='lower center',ncol=4);fig.subplots_adjust(bottom=.13);save(fig,out,'risk')
    fig,axes=plt.subplots(2,3,figsize=(12,7),sharex=True)
    for ax,(kind,a) in zip(axes.flat,[('matched',.7),('matched',1.),('matched',1.3),('shifted-t',1.),('shifted-normal',1.),('overlap',1.)]):
        sub=rs[(rs.kind==kind)&(rs.alpha==a)&(rs.n==300)]
        for m,c in zip(METHODS,COLORS):
            d=sub[sub.method==m];ax.plot(d.epsilon,d.median_abs_error,marker='o',ms=3,color=c,label=m,lw=1.4)
        ax.set_title(f'{kind}, alpha={a:g}');ax.set_xlabel('Contamination probability');ax.set_yscale('symlog',linthresh=.2);ax.set_ylabel('Median absolute error')
    handles,labels=axes[0,0].get_legend_handles_labels();fig.legend(handles,labels,loc='lower center',ncol=4);save(fig,out,'median_error')
    fig,axes=plt.subplots(1,2,figsize=(9,3.7));d=ag[(ag.n==300)&(ag.epsilon>0)]
    for (kind,a),g in d.groupby(['kind','alpha']):
        for ax,col in zip(axes,['auc','brier']):ax.plot(g.epsilon,g[col],marker='o',label=f'{kind}, {a:g}')
    axes[0].set_ylabel('ROC AUC');axes[1].set_ylabel('Brier score');[ax.set_xlabel('Contamination probability') for ax in axes];axes[1].legend(fontsize=7);save(fig,out,'allocation')
    fig,axes=plt.subplots(1,2,figsize=(9,3.7));m=alloc[(alloc.kind=='matched')&(alloc.n==300)]
    for a,g in m.groupby('alpha'):
        h=g.groupby('epsilon').alpha_hat.agg(['mean','sem']).reset_index();axes[0].errorbar(h.epsilon,h['mean'],yerr=h['sem'],marker='o',label=f'true alpha={a:g}');axes[0].axhline(a,ls=':',lw=.8,color='gray')
        h=g.groupby('epsilon').epsilon_hat.agg(['mean','sem']).reset_index();axes[1].errorbar(h.epsilon,h['mean'],yerr=h['sem'],marker='o',label=f'alpha={a:g}')
    axes[1].plot([0,.2],[0,.2],'k--',lw=1,label='Identity');axes[0].set_ylabel('Mean MAP shape');axes[1].set_ylabel('Mean MAP contamination fraction');[ax.set_xlabel('True contamination probability') for ax in axes];axes[0].legend(fontsize=8);save(fig,out,'recovery')
    progress.stage(1)
    coverage=[]; diag=[]
    for kind,a,n,e in [('matched',.7,100,.1),('matched',1.,300,.1),('matched',1.3,100,.2),('matched',1.,100,0.),('shifted-normal',1.,100,.1)]:
        progress.detail(f'{kind}, a={a:g}, n={n}, eps={e:g}')
        for rep in range(ncov):
            x,z=sample(rng,n,a,e,kind);ar,p,d=posterior(x,0,5,1,warm,draws,nextseed());auc,brier=metrics(z,p)
            for row in d.to_dict('records'):
                common=dict(kind=kind,alpha=a,n=n,epsilon=e,rep=rep);diag.append(dict(experiment='coverage',**common,**row));truth={'mu':0.,'theta':1.,'epsilon':e,'alpha':a}[row['parameter']]
                coverage.append(dict(**common,**row,truth=truth,covered=row['lo']<=truth<=row['hi'],width=row['hi']-row['lo'],auc=auc,brier=brier))
            progress.tick()
    cv=pd.DataFrame(coverage);cv.to_csv(out/'raw/coverage.csv',index=False);cv['error']=cv['mean']-cv.truth;cv['sq']=cv.error**2;cv['abs']=abs(cv.error)
    cs=cv.groupby(['kind','alpha','n','epsilon','parameter']).agg(reps=('covered','size'),coverage=('covered','mean'),width=('width','mean'),bias=('error','mean'),mse=('sq','mean'),mae=('abs','mean'),pass_rate=('diagnostic_pass','mean')).reset_index();cs['rmse']=np.sqrt(cs.mse);cs['mcse']=np.sqrt(cs.coverage*(1-cs.coverage)/cs.reps);zz=1.95996398454;nn=cs.reps;pp=cs.coverage;mid=(pp+zz**2/(2*nn))/(1+zz**2/nn);half=zz*np.sqrt(pp*(1-pp)/nn+zz**2/(4*nn**2))/(1+zz**2/nn);cs['wilson_lo']=mid-half;cs['wilson_hi']=mid+half;tab(cs,out,'coverage_summary')
    fig,ax=plt.subplots(figsize=(9,4));params=['theta','epsilon','alpha']; groups=list(cs.groupby(['kind','alpha','n','epsilon']));
    for j,par in enumerate(params):
        sub=[g[g.parameter==par].iloc[0] for _,g in groups]; vals=np.array([r.coverage for r in sub]);lo=np.array([r.wilson_lo for r in sub]);hi=np.array([r.wilson_hi for r in sub]);ax.errorbar(np.arange(len(groups))+(j-1)*.18,vals,yerr=np.vstack([vals-lo,hi-vals]).clip(0),fmt='o',label=par,capsize=3)
    ax.axhline(.95,color='k',ls='--',lw=1);ax.set_xticks(range(len(groups)),[f'{k[0]}\na={k[1]}, n={k[2]}\ne={k[3]}' for k,_ in groups]);ax.set_ylim(-.03,1.04);ax.set_ylabel('Coverage (Wilson 95% interval)');ax.legend();save(fig,out,'coverage')
    # Fixed-parameter score and finite-sample posterior stress.
    xx=np.linspace(-12,12,1001);fig,axes=plt.subplots(1,2,figsize=(9,3.4))
    for a in [.7,1.,1.3]:
        r=1-responsibilities(xx,np.array([0.,1.,.1,a]),0,5);lb,_=constants(a);axes[0].plot(xx,r,label=f'alpha={a:g}');axes[1].plot(xx,r*(-1/(2*a)+np.exp(lb)*abs(xx)**(2*a)),label=f'alpha={a:g}')
    axes[0].set_ylabel('Clean responsibility');axes[1].set_ylabel('Observed V-score');[ax.set_xlabel('Observation x') for ax in axes];axes[0].legend();save(fig,out,'score')
    progress.stage(2)
    xbase=rng.normal(size=119);tail=[];tilt_mean=np.nan
    for M in [0,2,4,8,16,32,64]:
        progress.detail(f"added point={M}")
        x=xbase if M==0 else np.r_[xbase,M];ar,p,d=posterior(x,0,5,1,warm,draws,nextseed());
        for row in d.to_dict('records'):diag.append(dict(experiment=f'tail_{M}',**row))
        th=d[d.parameter=='theta'].iloc[0]
        if M==0: tilt_mean=np.average(ar[:,:,1],weights=ar[:,:,2])
        tail.append(dict(M=M,mean=th['mean'],lo=th.lo,hi=th.hi,p_out=p[-1] if M else np.nan,tilted_theta_mean=tilt_mean,SV=np.var(x,ddof=1),GBPM=baseline(x)[2],MAD=baseline(x)[3]))
        progress.tick()
    tail=pd.DataFrame(tail);tab(tail,out,'tail');fig,axes=plt.subplots(1,2,figsize=(9,3.5));d=tail[tail.M>0];axes[0].plot(d.M,d['mean'],'o-',label='RBPM posterior');axes[0].fill_between(d.M.to_numpy(),d.lo.to_numpy(),d.hi.to_numpy(),alpha=.2);axes[0].axhline(tail.iloc[0]['mean'],color='k',ls=':',label='RBPM without point');axes[0].axhline(tilt_mean,color='#D55E00',ls='--',label='Epsilon-tilted reference')
    for m in ['SV','GBPM','MAD']:axes[0].plot(d.M,d[m],marker='.',label=m)
    axes[0].set_yscale('log');axes[0].set_ylabel('Variance estimate');axes[1].plot(d.M,d.p_out,'o-');axes[1].set_ylabel('Posterior contamination probability');axes[1].set_ylim(-.03,1.03);[ax.set_xscale('log',base=2) for ax in axes];[ax.set_xlabel('Added observation M') for ax in axes];axes[0].legend(fontsize=8);save(fig,out,'tail')
    # Wine: label construction, fits, sensitivity, repeated conditional resampling.
    progress.stage(3)
    wine=load_wine();j=wine.feature_names.index('flavanoids');pools=[wine.data[wine.target==i,j] for i in range(3)];truth=np.var(pools[0]);wrng=np.random.default_rng(seed); x=np.r_[pools[0],wrng.choice(pools[2],10,replace=False)];z=np.r_[np.zeros(59,int),np.ones(10,int)];perm=wrng.permutation(len(x));x,z=x[perm],z[perm]
    ar,p,wd=posterior(x,2.5,2,.25,warm,draws,nextseed());np.savez_compressed(out/'raw/wine_chains.npz',draws=ar,parameters=NAMES);tab(wd,out,'wine_posterior');auc,brier=metrics(z,p);tab(pd.DataFrame([dict(truth=truth,epsilon=z.mean(),auc=auc,brier=brier)]),out,'wine_classification');pd.DataFrame(dict(x=x,label=z,posterior_contamination=p)).to_csv(out/'raw/wine_observations.csv',index=False)
    for row in wd.to_dict('records'):diag.append(dict(experiment='wine',**row))
    progress.tick()
    fig,axes=plt.subplots(1,2,figsize=(9,3.6));
    for zz,label,c in [(0,'Target class 0','#0072B2'),(1,'Inserted class 2','#D55E00')]:
        axes[0].hist(x[z==zz],bins=np.linspace(x.min()-.05,x.max()+.05,18),alpha=.55,label=label,color=c);axes[1].scatter(x[z==zz],p[z==zz],label=label,color=c,s=25,alpha=.8)
    axes[0].set_ylabel('Count');axes[1].set_ylabel('Posterior contamination probability');[ax.set_xlabel('Flavanoids') for ax in axes];axes[0].legend();save(fig,out,'wine_allocation')
    fit,ok=mapfit(x,2.5,2,.25); th=wd[wd.parameter=='theta'].iloc[0]
    comp=pd.DataFrame(dict(method=METHODS+['RBPM posterior'],estimate=baseline(x,.25)+[fit[1],th['mean']],lo=[np.nan]*7+[th.lo],hi=[np.nan]*7+[th.hi]));tab(comp,out,'wine_method_comparison')
    fig,ax=plt.subplots(figsize=(7,4));yy=np.arange(len(comp));ax.scatter(comp.estimate,yy,color='#0072B2',zorder=3);ax.hlines(yy[-1],th.lo,th.hi,color='#D55E00',lw=2);ax.axvline(truth,color='k',ls='--',label='Empirical clean variance');ax.set_yticks(yy,comp.method);ax.invert_yaxis();ax.set_xlabel('Variance estimate');ax.legend(fontsize=8);save(fig,out,'wine_methods')
    fig,axes=plt.subplots(4,2,figsize=(10,9));
    for j,par in enumerate(NAMES):
        for c in range(4):axes[j,0].plot(ar[c,:,j],lw=.5,alpha=.8);axes[j,1].hist(ar[c,:,j],bins=30,density=True,histtype='step',lw=1)
        axes[j,0].set_ylabel(par);axes[j,1].set_xlabel(par)
    axes[-1,0].set_xlabel('Retained iteration');save(fig,out,'wine_chains')
    progress.stage(4)
    sens=[]
    for scale in [1.,2.,4.]:
        for rv in [.0625,.25,1.]:
            progress.detail(f'scale={scale:g}, reference={rv:g}')
            for row in posterior(x,2.5,scale,rv,warm,draws,nextseed())[2].to_dict('records'):sens.append(dict(scale=scale,refvar=rv,**row));diag.append(dict(experiment=f'wine_sensitivity_{scale}_{rv}',**row))
            progress.tick()
    sens=pd.DataFrame(sens);tab(sens,out,'wine_sensitivity');fig,ax=plt.subplots(figsize=(7,3.8));d=sens[sens.parameter=='theta'];
    for i,(rv,g) in enumerate(d.groupby('refvar')):ax.errorbar(g.scale+(i-1)*.05,g['mean'],yerr=np.vstack([g['mean']-g.lo,g.hi-g['mean']]).clip(0),fmt='o-',label=f'prior reference={rv:g}',capsize=3)
    ax.axhline(truth,color='k',ls='--',label='Empirical target');ax.set_xlabel('Working contaminant scale');ax.set_ylabel('Posterior clean variance');ax.legend(fontsize=8);save(fig,out,'wine_sensitivity')
    progress.stage(5)
    wb=[];wa=[]
    for cls in [1,2]:
        progress.detail(f'inserted class={cls}')
        for rep in range(nboot):
            xx=np.r_[rng.choice(pools[0],50),rng.choice(pools[cls],8)];zz=np.r_[np.zeros(50,int),np.ones(8,int)];idx=rng.permutation(58);xx,zz=xx[idx],zz[idx];fit,ok=mapfit(xx,2.5,2,.25);pp=responsibilities(xx,fit,2.5,2);auc,brier=metrics(zz,pp);wa.append(dict(inserted_class=cls,rep=rep,auc=auc,brier=brier,optimizer_success=ok))
            for name,val in zip(METHODS,baseline(xx,.25)+[fit[1]]):wb.append(dict(inserted_class=cls,rep=rep,method=name,estimate=val,truth=truth))
            progress.tick()
    progress.finish()
    wb=pd.DataFrame(wb);wa=pd.DataFrame(wa);wb.to_csv(out/'raw/wine_resampling.csv',index=False);wa.to_csv(out/'raw/wine_resampling_allocations.csv',index=False);ws=risk_summary(wb,['inserted_class','method']);tab(ws,out,'wine_resampling_summary');tab(wa.groupby('inserted_class').agg(auc=('auc','mean'),brier=('brier','mean'),optimizer_success=('optimizer_success','mean')).reset_index(),out,'wine_resampling_allocation')
    fig,axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
    for ax,cls in zip(axes,[1,2]):
        ax.boxplot([wb[(wb.inserted_class==cls)&(wb.method==m)].estimate for m in METHODS],tick_labels=METHODS,showfliers=True,flierprops={'markersize':2,'alpha':.4});ax.axhline(truth,ls='--',color='k');ax.tick_params(axis='x',rotation=45);ax.set_title(f'Inserted cultivar class {cls}');ax.set_ylabel('Variance estimate')
    save(fig,out,'wine_resampling')
    di=pd.DataFrame(diag);di.to_csv(out/'raw/all_diagnostics.csv',index=False)
    # Output LaTeX only for research mode. Quick results cannot silently become manuscript evidence.
    if mode=='research':
        d=rs[(rs.kind=='matched')&(rs.alpha==1)&(rs.n==300)&(rs.epsilon==.1)]
        s=f'The risk experiment used {nr} replications per configuration, and the coverage experiment used {ncov} per configuration. All {len(di)} parameter-level diagnostic records are provided in the computational supplement; the fraction meeting both diagnostic thresholds was {di.diagnostic_pass.mean():.3f}. Numerical conclusions must account for any failed diagnostics.\n'
        s+=tabletex(d,['method','bias','rmse','median_abs_error','mae'],'Matched design: Gaussian clean law, $n=300$, contamination probability 0.1.',['Method','Bias','RMSE','Median AE','MAE'])
        s+=tabletex(cs[cs.parameter=='theta'],['kind','alpha','n','epsilon','coverage','width','pass_rate'],'Clean-variance interval coverage, mean width and diagnostic pass fraction.',['Design','Shape','n','Fraction','Coverage','Width','Pass rate'])
        s+=figtex('risk','Scale risk at $n=300$; error bars are one descriptive Monte Carlo standard error where displayed; omitted for contaminated polynomial-tail designs. A symmetric-log vertical scale preserves large errors and the near-zero region.')+figtex('median_error','Median absolute estimation error, a complementary summary that remains meaningful when squared-error risk is infinite.')+figtex('allocation','Allocation discrimination and probability calibration for the MAP fits. The overlap case has no observation-level label signal in the population.')+figtex('recovery','Recovery of shape and contamination fraction in the matched design; bars show descriptive Monte Carlo standard errors.')+figtex('coverage','Equal-tailed interval coverage across selected scenarios, with Wilson Monte Carlo intervals. At zero contamination the continuous prior excludes the exact boundary from equal-tailed intervals.')+figtex('score','Fixed-parameter conditional responsibilities and direct clean-scale scores.')+figtex('tail','Finite-sample posterior sensitivity to an added point; shading shows equal-tailed 95\\% credible intervals. The epsilon-tilted reference is computed from the posterior without the added point.')
        (out/'simulation.tex').write_text(s)
        primary=pd.read_csv(out/'tables/wine_classification.csv').iloc[0];s=f'The empirical target variance is {truth:.5f}. The primary benchmark has AUC {primary.auc:.3f} and Brier score {primary.brier:.3f}. The repeated benchmark uses {nboot} resamples per inserted class.\n'
        s+=tabletex(wd,['parameter','mean','lo','hi','rhat','ess_bulk','ess_tail'],'Primary Wine posterior and convergence diagnostics.',['Parameter','Mean','2.5%','97.5%','Rhat','Bulk ESS','Tail ESS'])+figtex('wine_allocation','Primary benchmark: observed values and integrated posterior contamination probabilities.')+figtex('wine_methods','Point estimates and the RBPM equal-tailed posterior interval; dashed line is the empirical target.')+figtex('wine_chains','Four retained chains and chain-specific marginal histograms.')+figtex('wine_sensitivity','Sensitivity of the posterior clean variance to the working contaminant scale and prior reference variance.')+figtex('wine_resampling','Repeated real-data mixtures; dashed line denotes the empirical class-0 variance.')
        s+=tabletex(ws[ws.inserted_class==2],['method','bias','rmse','mae','median_abs_error'],'Repeated Wine benchmark, inserted class 2.',['Method','Bias','RMSE','MAE','Median AE'])
        (out/'wine.tex').write_text(s)
    else:
        for f in ['simulation.tex','wine.tex']:
            (out/f).unlink(missing_ok=True)
    import scipy,sklearn,numba
    manifest=dict(mode=mode,seed=seed,risk_reused=risk_reused,risk_reps=nr,coverage_reps=ncov,wine_reps=nboot,warmup=warm,draws_per_chain=draws,chains=4,seconds=time.time()-start,python=platform.python_version(),versions={m.__name__:m.__version__ for m in [np,pd,scipy,sklearn,numba,az,matplotlib]},diagnostic_pass_fraction=float(di.diagnostic_pass.mean()),map_success_fraction=float(alloc.optimizer_success.mean()),note='Quick outputs are software checks, not scientific results.' if mode=='quick' else 'Research results require review of diagnostics and numerical findings before submission.')
    (out/'manifest.json').write_text(json.dumps(manifest,indent=2));shutil.make_archive(str(out.resolve()),'zip',root_dir=out.parent,base_dir=out.name);print(json.dumps(manifest,indent=2),flush=True);print('Finished. Results ZIP: '+str(out.resolve())+'.zip',flush=True)


In [4]:
# Fail early if the actual posterior/diagnostic path is incompatible.
# This uses an independent seed and does not change the research simulation RNG.
print("Checking numerical libraries and posterior diagnostics...", flush=True)
for value in (1.0, np.array(1.0), np.array([1.0]), np.array([[1.0]])):
    assert diagnostic_scalar(value, "preflight") == 1.0
x_test, _ = sample(np.random.default_rng(90210), 100, 1.0, 0.1, "matched")
a_test, p_test, d_test = posterior(x_test, 0.0, 5.0, 1.0, 200, 250, 90211)
assert a_test.shape == (4, 250, 4)
assert np.isfinite(a_test).all() and np.isfinite(p_test).all()
assert np.isfinite(d_test[["rhat", "ess_bulk", "ess_tail", "mcse"]].to_numpy()).all()
print(d_test[["parameter", "rhat", "ess_bulk", "ess_tail", "mcse"]].to_string(index=False))
print("Preflight passed. Short test chains are not intended for scientific inference.", flush=True)


Checking numerical libraries and posterior diagnostics...
parameter     rhat   ess_bulk   ess_tail     mcse
       mu 1.049197 129.137121 142.220063 0.011273
    theta 1.029222 148.042763 157.132084 0.015834
  epsilon 1.018180 196.567927 184.228286 0.002071
    alpha 1.031542 113.838098 213.409927 0.017572
Preflight passed. Short test chains are not intended for scientific inference.


In [5]:
from pathlib import Path
run(MODE, Path("results"), SEED_RUN, resume_risk=False)


Benchmarking CPU speed for an initial runtime estimate...
Measured seconds per task (initial): risk=0.033, posterior=0.749, wine=0.031
Initial estimated computational runtime: 7m 36s (excludes plotting, setup and ZIP; adapts during execution)


Overall computational tasks:   0%|          | 0/6917 [00:00]

1/6 Risk simulations:   0%|          | 0/6000 [00:00<?, ?rep/s]

2/6 Posterior coverage:   0%|          | 0/300 [00:00<?, ?rep/s]

3/6 Extreme-observation analysis:   0%|          | 0/7 [00:00<?, ?rep/s]

4/6 Wine primary posterior:   0%|          | 0/1 [00:00<?, ?rep/s]

5/6 Wine sensitivity:   0%|          | 0/9 [00:00<?, ?rep/s]

6/6 Wine resampling:   0%|          | 0/600 [00:00<?, ?rep/s]

All computational tasks complete. Finalizing figures, tables and ZIP.
{
  "mode": "research",
  "seed": 20260816,
  "risk_reused": false,
  "risk_reps": 200,
  "coverage_reps": 60,
  "wine_reps": 300,
  "warmup": 1000,
  "draws_per_chain": 2000,
  "chains": 4,
  "seconds": 504.3507356643677,
  "python": "3.13.18",
  "versions": {
    "numpy": "2.4.6",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "sklearn": "1.6.1",
    "numba": "0.64.0",
    "arviz": "0.22.0",
    "matplotlib": "3.10.0"
  },
  "diagnostic_pass_fraction": 0.9810725552050473,
  "map_success_fraction": 1.0,
  "note": "Research results require review of diagnostics and numerical findings before submission."
}
Finished. Results ZIP: /content/results.zip


In [7]:
from google.colab import files
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>